# Step 5: Answering the Business Questions in SQL
The cleaned data is loaded into a SQLite database and the key questions are re-answered with SQL from [`sql/churn_analysis.sql`](../sql/churn_analysis.sql).
The queries use CTEs, CASE expressions, joins and window functions (`NTILE`, `RANK`). Results are checked against the pandas analysis.

In [1]:
import sqlite3
import pandas as pd
from pathlib import Path

PROC = Path('../data/processed')
df = pd.read_pickle(PROC / 'policy_terms_clean.pkl')
cols = ['Policy_Term_id', 'Client_id', 'Carrier_id', 'Province', 'Line_of_Business', 'LOB_Group', 'bundled',
        'Term_Premium', 'premium_outlier', 'tenure_years', 'start_year', 'CAN_REN', 'CAN_MT', 'churned']
con = sqlite3.connect(PROC / 'insurance.db')
df[cols].to_sql('policy_terms', con, if_exists='replace', index=False)
print(con.execute('SELECT COUNT(*) FROM policy_terms').fetchone()[0], 'rows loaded')

99999 rows loaded


In [2]:
sql = Path('../sql/churn_analysis.sql').read_text()
queries = [q.strip() for q in sql.split(';') if 'SELECT' in q]
results = {}
for q in queries:
    title = next(l for l in q.splitlines() if l.startswith('-- Q') or l.startswith('-- Impact'))[3:]
    results[title.split(':')[0]] = pd.read_sql(q, con)
print(list(results))

['Q1', 'Q2', 'Q3', 'Q4', 'Q5', 'Q6a', 'Q6b', 'Impact']


In [3]:
results['Q1']

,terms,renewal_cancel_pct,midterm_cancel_pct,churn_pct
0,99999,10.8,6.3,17.1


In [4]:
results['Q2']

,Line_of_Business,terms,churn_pct
0,TPAC,7622,22.5
1,MOTO,299,19.4
2,RDWL,372,19.1
3,AUTO,57114,18.0
4,CNDO,4107,16.4
5,BOAT,240,15.8
6,HOME,29151,14.2
7,TRAI,839,12.3


In [5]:
results['Q3']

,bundle_status,LOB_Group,terms,churn_pct
0,Auto + Property,Auto,12655,14.4
1,Single line,Auto,44758,19.0
2,Auto + Property,Property,12714,15.0
3,Single line,Property,29872,16.3


In [6]:
results['Q4']

,tenure_band,terms,churn_pct
0,1) <1 yr,41298,15.8
1,2) 1-3 yrs,17282,24.4
2,3) 3-5 yrs,13439,20.8
3,4) 5-10 yrs,20978,13.5
4,5) 10+ yrs,7002,10.5


In [7]:
results['Q5']

,premium_quartile,terms,min_premium,max_premium,churn_pct
0,1,25000,0.21,1042.00,17.0
1,2,25000,1042.00,1662.00,15.1
2,3,25000,1662.00,2524.00,16.6
3,4,24999,2524.00,2874713.17,19.7


In [8]:
results['Q6a']

,Province,terms,churn_pct
0,NB,7319,32.5
1,AB,27301,19.6
2,NS,2467,18.8
3,ON,59892,14.2
4,BC,1729,12.6


In [9]:
results['Q6b']

,Province,Carrier_id,terms,churn_pct,churn_rank
0,AB,65575,2961,48.3,1
1,AB,65581,10102,18.0,2
2,AB,65577,8134,16.6,3
3,AB,65568,597,13.7,4
4,AB,65582,2504,11.5,5
5,AB,65569,2159,10.8,6
6,NB,65575,3342,49.1,1
7,NB,65569,1457,27.7,2
8,NB,65582,529,16.4,3
9,NB,65577,647,13.1,4


In [10]:
results['Impact']

,segment_terms,lost_terms,share_of_all_cancellations_pct,avg_lost_premium,terms_saved_at_10pct,premium_retained_at_10pct
0,33028,6817,39.8,2686.0,682.0,1830864.0


## Cross-check with pandas

In [11]:
assert results['Q1'].loc[0, 'churn_pct'] == round(df['churned'].mean() * 100, 1)
assert results['Impact'].loc[0, 'lost_terms'] == 6817
nb_rate = results['Q6a'].set_index('Province').loc['NB', 'churn_pct']
assert nb_rate == round(df.loc[df.Province == 'NB', 'churned'].mean() * 100, 1)
con.close()
print('SQL results match the pandas analysis')

SQL results match the pandas analysis
